In [2]:
import pandas as pd

In [5]:

footfall_df = pd.read_csv('../datasets/raw/StationFootfall_2024_2025.csv')

footfall_df.rename(columns={'TravelDate' : 'date', 'DayOfWeek' : 'day_of_week', 'Station' : 'station','EntryTapCount' : 'entry_tap_count', 'ExitTapCount' : 'exit_tap_count'}, inplace=True)

footfall_df['date'] = pd.to_datetime(footfall_df['date'].astype(str), format="%Y%m%d")

footfall_df['baseline_entry'] = (
    footfall_df.groupby(['station', 'day_of_week'])['entry_tap_count']
      .transform('mean')
)

footfall_df['baseline_exit'] = (
    footfall_df.groupby(["station", 'day_of_week'])['exit_tap_count']
      .transform('mean')
)

footfall_df['overcrowding_index'] = (
    footfall_df['entry_tap_count'] / footfall_df['baseline_entry']
) * 100

print(footfall_df.head())

        date day_of_week          station  entry_tap_count  exit_tap_count  \
0 2024-01-01      Monday   Abbey Road DLR              395             375   
1 2024-01-01      Monday       Abbey Wood             5898            5963   
2 2024-01-01      Monday    Acton Central              609             474   
3 2024-01-01      Monday  Acton Main Line             1717            1710   
4 2024-01-01      Monday       Acton Town             2928            3334   

   baseline_entry  baseline_exit  overcrowding_index  
0      859.721154     812.134615           45.945130  
1    15872.230769   15026.711538           37.159238  
2     2093.817308    2051.269231           29.085632  
3     3833.223301    3772.378641           44.792590  
4     6588.625000    6825.192308           44.440228  


In [6]:
weather_df = pd.read_csv('../datasets/raw/open-meteo-51.49N0.49W24m.csv')
weather_df['date'] = pd.to_datetime(weather_df['time'], format="%Y-%m-%d")

In [8]:
df_combined = pd.merge(
    footfall_df,
    weather_df,
    on="date",
    how="left"
)

In [10]:
df_combined.to_csv('../datasets/raw/combined.csv', index=False)

In [11]:
df_combined.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 311447 entries, 0 to 311446
Data columns (total 26 columns):
 #   Column                           Non-Null Count   Dtype         
---  ------                           --------------   -----         
 0   date                             311447 non-null  datetime64[ns]
 1   day_of_week                      311447 non-null  object        
 2   station                          311447 non-null  object        
 3   entry_tap_count                  311447 non-null  int64         
 4   exit_tap_count                   311447 non-null  int64         
 5   baseline_entry                   311447 non-null  float64       
 6   baseline_exit                    311447 non-null  float64       
 7   overcrowding_index               311447 non-null  float64       
 8   time                             311447 non-null  object        
 9   temperature_2m_max (°C)          311447 non-null  float64       
 10  temperature_2m_min (°C)          311447 non-

In [22]:
df_combined['is_raining'] = (df_combined['rain_sum (mm)'] > 0.0).astype('int')

In [23]:
df_combined['is_raining']

0         1
1         1
2         1
3         1
4         1
         ..
311442    0
311443    0
311444    0
311445    0
311446    0
Name: is_raining, Length: 311447, dtype: int64

In [24]:
df_combined = df_combined.rename(columns={
    "date": "date",
    "day_of_week": "day_of_week",
    "station": "station",
    "entry_tap_count": "entries",
    "exit_tap_count": "exits",
    "baseline_entry": "baseline_entries",
    "baseline_exit": "baseline_exits",
    "overcrowding_index": "overcrowding",
    "time": "hour",
    "temperature_2m_max (°C)": "temp_max",
    "temperature_2m_min (°C)": "temp_min",
    "temperature_2m_mean (°C)": "temp_mean",
    "apparent_temperature_mean (°C)": "app_temp_mean",
    "apparent_temperature_max (°C)": "app_temp_max",
    "apparent_temperature_min (°C)": "app_temp_min",
    "wind_speed_10m_max (km/h)": "wind_max",
    "wind_gusts_10m_max (km/h)": "wind_gust_max",
    "wind_direction_10m_dominant (°)": "wind_dir",
    "rain_sum (mm)": "rain_mm",
    "sunshine_duration (s)": "sunshine_s",
    "daylight_duration (s)": "daylight_s",
    "sunrise (iso8601)": "sunrise",
    "sunset (iso8601)": "sunset",
    "precipitation_hours (h)": "precip_hours",
    "snowfall_sum (cm)": "snow_cm",
    "precipitation_sum (mm)": "precip_mm"
})

In [29]:
df_combined.to_csv('../datasets/raw/combined.csv', index=False)

In [30]:
df = pd.read_csv('../datasets/raw/combined.csv')

In [32]:
# convert date time
df['date'] = pd.to_datetime(df['date'])
df['hour'] = pd.to_datetime(df['hour'], format="%H:%M", errors='coerce').dt.hour

# useful time features
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day

# sunrise and sunset
df['sunrise_hour'] = pd.to_datetime(df['sunrise'], errors='coerce').dt.hour
df['sunset_hour'] = pd.to_datetime(df['sunset'], errors='coerce').dt.hour

# encode categories
df['station_code'] = df['station'].astype('category').cat.codes

# numerical day of week
if df['day_of_week'].dtype == object:
    df['day_of_week_code'] = df['day_of_week'].astype('category').cat.codes
else:
    df['day_of_week_code'] = df['day_of_week']

# drop things not needed for random forest
df_rf = df.drop(columns=[
    'station', 'day_of_week', 'sunrise', 'sunset', 'date'
])

# missing vals
df_rf = df_rf.fillna(0)

   entries  exits  baseline_entries  baseline_exits  overcrowding  hour  \
0      395    375        859.721154      812.134615     45.945130   0.0   
1     5898   5963      15872.230769    15026.711538     37.159238   0.0   
2      609    474       2093.817308     2051.269231     29.085632   0.0   
3     1717   1710       3833.223301     3772.378641     44.792590   0.0   
4     2928   3334       6588.625000     6825.192308     44.440228   0.0   

   temp_max  temp_min  temp_mean  app_temp_mean  ...  precip_hours  snow_cm  \
0      11.2       6.2        7.9            3.5  ...           9.0      0.0   
1      11.2       6.2        7.9            3.5  ...           9.0      0.0   
2      11.2       6.2        7.9            3.5  ...           9.0      0.0   
3      11.2       6.2        7.9            3.5  ...           9.0      0.0   
4      11.2       6.2        7.9            3.5  ...           9.0      0.0   

   precip_mm  is_raining  month  day  sunrise_hour  sunset_hour  station_c